<a href="https://colab.research.google.com/github/MMujtabaX/agentic-ai-journey/blob/main/04_langgraph_routing/langgraph_routing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lecture 1 : LangGraph Fundamentals
**Bootcamp Track:** Agentic AI → LangGraph  
**Domain Use Case:** *Internal AI Enablement Advisor for an SME* (Strategy vs Implementation routing)

---

## What you’ll build today
A small but **production-style** LangGraph that:
1. Accepts an internal “AI adoption” query from an employee
2. Classifies whether it’s a **Strategy** question or an **Implementation** request
3. Routes deterministically to the correct node
4. Produces **decision-grade** outputs (strategy) or a **phased plan** (implementation)
5. Keeps an explicit state trace you can inspect

---

## Why LangGraph
**LangGraph is a framework for building LLM applications as explicit, stateful graphs**—so you can control flow, branch, retry, and audit execution.

---

## How it works
LangGraph is built around four core ideas:

### 1) **State**
A typed dictionary that holds the “truth” of the system at any moment.
- If it’s not in state, the graph can’t use it.

### 2) **Nodes**
Pure functions that take `state → updates`.
- Each node is a component: router, planner, verifier, retriever, etc.

### 3) **Edges**
Connections between nodes that define execution order.

### 4) **Conditional Edges**
A router function chooses the next node based on state.

---

## Diagram of today’s graph
```text
                ┌──────────────────────────┐
Employee Query ─► Business Intent Analyzer  │
                └───────────┬──────────────┘
                            │
        ┌───────────────────┼────────────────────┐
        │                   │                    │
Strategy Question     Implementation Request   Ambiguous
        │                   │                    │
┌───────▼─────────┐  ┌──────▼────────────┐       ▼
│ AI Strategy Node │  │ AI Implementation │      END
│ (Business Lens)  │  │ Planner Node      │
└───────┬─────────┘  └────────┬──────────┘
        │                      │
        └───────────────► END ◄┘
```

---

## Expected skill outcomes
By the end, you can:
- Explain LangGraph components (State/Nodes/Edges/Conditionals)
- Write a **domain-specific** router node with strong prompting
- Compile and run a graph deterministically
- Inspect state evolution for debugging

# 0) Setup: Install dependencies
We’ll use:
- `langgraph` for graph orchestration
- `langchain-openai` for an LLM wrapper (model-agnostic conceptually)
- `pydantic` (optional) for future validation patterns

> **Note (Teaching):** LangGraph works with other providers too (Groq, Anthropic, etc.). OpenAI is used here for minimal setup.

In [ ]:
!pip install -U langgraph langchain langchain-openai pydantic -q

In [ ]:
import os
from typing import TypedDict, List, Literal

from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END

## 1.1 Set your API key
If you're using Colab, prefer:
- **Runtime → Secrets** (or environment variables)

For quick demos, set it like this:

In [ ]:
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")


# 2) Create the LLM
We initialize the LLM once and reuse it across nodes.

- `temperature=0.2` keeps responses stable (useful for deterministic systems)
- `model="gpt-4o-mini"` is fast and inexpensive for teaching

In [ ]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.2
)

# 3) Define the **domain-specific State**
This is the most important LangGraph concept.

Our **AdvisorState** stores:
- `query`: the internal employee request
- `intent`: the routing label (strategy/implementation/unknown)
- `department`: optional metadata (helps personalization later)
- `response_log`: a trace of what the system decided and produced

> **Teaching line:** *“State is explicit memory. Nodes can only use what state contains.”*

In [ ]:
class AdvisorState(TypedDict):
    query: str
    intent: Literal["strategy", "implementation", "unknown"]
    department: str
    response_log: List[str]

# 4) Node 1 — Business Intent Analyzer (Router Controller)
This node **does not answer** the user.
It only classifies the query into a routing intent.

Why this matters:
- In many real systems, the router is the “traffic controller”.
- If routing is weak, the whole system is unreliable.

### Output contract
This node must return **exactly one** label:
- `strategy` → leadership/ROI/risks/adoption decisions
- `implementation` → execution steps, architecture, tooling
- `unknown` → ambiguous, incomplete, off-topic

In [ ]:
def classify_business_intent(state: AdvisorState):
    system_prompt = """
You are an internal AI Enablement Advisor router for a mid-sized company.
Your job is to classify internal AI-related queries for deterministic routing.
You do not answer the query. You only return a label.
"""

    task_prompt = f"""
Classify the following internal query into EXACTLY ONE category:

- strategy → leadership-level, business impact, ROI, risk, adoption decisions
- implementation → execution details, tooling, architecture, steps, deployment
- unknown → unclear, incomplete, or unrelated

Query:
\"\"\"{state['query']}\"\"\"

Rules:
- Respond with only one label: strategy / implementation / unknown
- No punctuation
- No explanations
"""

    intent = llm.invoke(system_prompt + task_prompt).content.strip().lower()

    # Safety: normalize unexpected outputs
    if intent not in ["strategy", "implementation", "unknown"]:
        intent = "unknown"

    return {
        "intent": intent,
        "response_log": state["response_log"] + [f"[Router] intent={intent}"]
    }

# 5) Node 2 — AI Strategy Advisor (Decision-Grade Output)
This node is used when the query is **strategy**.

We want leadership-friendly output:
- Executive takeaway
- Business impact
- Risks & constraints
- Recommendation
- Clarifying questions (only if needed)

> **Teaching note:** This is where you show students that “agentic” systems can still be controlled & structured.

In [ ]:
def ai_strategy_node(state: AdvisorState):
    system_prompt = """
You are a senior AI strategy advisor for an SME.
You advise leadership on AI adoption decisions.
Focus on business value, feasibility, risk, trade-offs, and decision clarity.
Avoid hype and avoid deep implementation detail unless asked.
If something is unknown, say what needs to be clarified.
"""

    task_prompt = f"""
Answer the internal strategy question below.

Output format:
1) Executive takeaway (1-2 lines)
2) Business impact (bullets)
3) Risks & constraints (bullets)
4) Recommendation (clear next step)
5) 3 clarifying questions (only if needed)

Internal Query:
\"\"\"{state['query']}\"\"\"
"""

    response = llm.invoke(system_prompt + task_prompt)

    return {
        "response_log": state["response_log"] + [response.content]
    }

# 6) Node 3 — AI Implementation Planner (Phased Plan)
This node is used when the query is **implementation**.

We produce a realistic rollout plan suitable for SMEs:
- Phases (0..3)
- Tech stack suggestions
- Risks + mitigations
- A checklist of what we need from the company

> **Teaching note:** This style is ideal for internal delivery teams and clients.

In [ ]:
def ai_implementation_node(state: AdvisorState):
    system_prompt = """
You are an AI implementation planner inside an enterprise environment.
Your job is to produce a realistic plan with phases, dependencies, and risks.
Do not invent company-specific details. If missing info, ask targeted questions.
"""

    task_prompt = f"""
Create a practical implementation plan for the request below.

Constraints:
- Assume SME resources (small team, limited infra)
- Prefer simple, testable milestones
- Include security/privacy notes when relevant

Output format:
A) Summary (2-3 lines)
B) Phase plan (Phase 0..3 with bullets)
C) Tech stack suggestion (bullets)
D) Risks + mitigations (bullets)
E) What you need from the company (checklist)

Internal Request:
\"\"\"{state['query']}\"\"\"
"""

    response = llm.invoke(system_prompt + task_prompt)

    return {
        "response_log": state["response_log"] + [response.content]
    }

# 7) Router Function — Conditional Edges
This function reads `state["intent"]` and decides the next node.

This is the key difference from “free-form agent loops”:
- **We** decide the legal transitions.
- The model doesn’t get to randomly jump around.

Routing rules:
- `strategy` → `ai_strategy`
- `implementation` → `ai_implementation`
- otherwise → `END`

In [ ]:
def route_by_business_intent(state: AdvisorState):
    if state["intent"] == "strategy":
        return "ai_strategy"
    elif state["intent"] == "implementation":
        return "ai_implementation"
    else:
        return END

# 8) Build the Graph (Nodes + Edges)
Now we assemble:
1. Create `StateGraph(AdvisorState)`
2. Add nodes (functions)
3. Set entry point
4. Add conditional edges (routing)
5. Add terminal edges to `END`

> **Important:** A graph definition is not executable until we **compile** it.

In [ ]:
graph = StateGraph(AdvisorState)

# Add nodes
graph.add_node("intent_classifier", classify_business_intent)
graph.add_node("ai_strategy", ai_strategy_node)
graph.add_node("ai_implementation", ai_implementation_node)

# Entry point
graph.set_entry_point("intent_classifier")

# Conditional routing
graph.add_conditional_edges(
    "intent_classifier",
    route_by_business_intent
)

# End edges
graph.add_edge("ai_strategy", END)
graph.add_edge("ai_implementation", END)

# 9) Compile the Graph
Compilation produces an executable artifact.

> **Teaching line:** *“Declare → Compile → Execute.”*

If students get `NameError: advisor not defined`, it means they skipped this cell.

In [ ]:
advisor = graph.compile()

# 11) Run Example 1 — Strategy Query
This should route:
`intent_classifier → ai_strategy → END`

We also print the `response_log` so students can see:
- the routing decision
- the final output

In [ ]:
state = {
    "query": "Should we adopt AI for our customer support operations this year?",
    "intent": "unknown",
    "department": "operations",
    "response_log": []
}

result = advisor.invoke(state)

for item in result["response_log"]:
    print("\n" + "="*80)
    print(item)

print("\nFinal intent:", result.get("intent"))


[Router] intent=strategy

1) **Executive takeaway:** Adopting AI for customer support this year can enhance efficiency and customer satisfaction, but careful consideration of implementation risks and resource allocation is essential.

2) **Business impact:**
   - Improved response times and availability, leading to enhanced customer satisfaction.
   - Increased operational efficiency by automating routine inquiries, allowing human agents to focus on complex issues.
   - Potential cost savings through reduced staffing needs and improved resource allocation.
   - Data-driven insights from AI interactions can inform product and service improvements.

3) **Risks & constraints:**
   - Initial investment costs and ongoing maintenance expenses may strain budgets.
   - Potential resistance from staff and customers who may prefer human interaction.
   - Data privacy and compliance risks associated with handling customer information.
   - Dependence on technology could lead to service disruptio

# 12) Run Example 2 — Implementation Request
This should route:
`intent_classifier → ai_implementation → END`

In [ ]:
state = {
    "query": "How can we build an internal RAG system for HR policies?",
    "intent": "unknown",
    "department": "HR",
    "response_log": []
}

result = advisor.invoke(state)

for item in result["response_log"]:
    print("\n" + "="*80)
    print(item)

print("\nFinal intent:", result.get("intent"))


[Router] intent=implementation

### A) Summary
The implementation plan outlines a phased approach to develop an internal Retrieval-Augmented Generation (RAG) system for HR policies, focusing on leveraging existing resources while ensuring security and privacy compliance. The plan emphasizes simple milestones to facilitate testing and validation.

### B) Phase Plan

**Phase 0: Requirements Gathering and Planning**
- Identify key stakeholders and form a small project team.
- Define the scope of the RAG system, including specific HR policies to be included.
- Gather requirements for data sources, user access, and security/privacy considerations.
- Create a project timeline with milestones.

**Phase 1: Data Collection and Preparation**
- Collect existing HR policy documents and related materials.
- Organize and preprocess data for ingestion into the RAG system (e.g., text cleaning, formatting).
- Ensure compliance with data privacy regulations (e.g., GDPR, HIPAA) during data handling.

**

# 13) Run Example 3 — Ambiguous Query (Safe Stop)
If the classifier returns `unknown`, the router ends safely.

> **Teaching note:** In production, you might route `unknown` to a clarifier node instead.

In [ ]:
state = {
    "query": "What is elon musk's wife name?",
    "intent": "unknown",
    "department": "unknown",
    "response_log": []
}

result = advisor.invoke(state)

for item in result["response_log"]:
    print("\n" + "="*80)
    print(item)

print("\nFinal intent:", result.get("intent"))


[Router] intent=unknown

Final intent: unknown
